<a href="https://colab.research.google.com/github/Pranshu244/Student-Risk-Detection-System/blob/main/Website_Code/student_risk_detection_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit

In [ ]:
!pip install pyngrok

In [ ]:
%%writefile student_risk_detection_app.py
import streamlit as st
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
import tempfile
import os
import base64

def get_base64_of_bin_file(bin_file):
    with open(bin_file, 'rb') as f:
        data = f.read()
    return base64.b64encode(data).decode()
logo_base64 = get_base64_of_bin_file("logo.png")
st.markdown(f"""<div style="text-align:center;"> <img src="data:image/png;base64,{logo_base64}" width="200"></div>""",unsafe_allow_html=True)
st.markdown("<h1 style='text-align:center;font-weight:bold;font-style:italic;color:#2E7D32;'>Student Risk Detection System</h1>", unsafe_allow_html=True)
st.markdown("<p style='text-align:center;font-style:italic;color:#558B2F;'>AI-based academic risk screening</p>", unsafe_allow_html=True)

uploaded_model = st.file_uploader("Upload model.pkl", type="pkl")
uploaded_ae = st.file_uploader("Upload autoencoder.keras", type=["keras", "h5"])

if uploaded_model and uploaded_ae:
    dataset_scaler, iso, kmeans, cluster_risk_map, ae_threshold = pickle.load(uploaded_model)
    try:
        feature_cols = list(dataset_scaler.feature_names_in_)
    except AttributeError:
        st.error("Scaler missing feature_names_in_. Ensure you fit it on dataset features before dumping.")

    with tempfile.TemporaryDirectory() as tmpdir:
        ae_path = os.path.join(tmpdir, "autoencoder.keras")
        with open(ae_path, "wb") as f:
            f.write(uploaded_ae.getbuffer())
        autoencoder = tf.keras.models.load_model(ae_path, compile=False)

    st.session_state.update({
        "scaler": dataset_scaler,
        "iso": iso,
        "kmeans": kmeans,
        "cluster_risk_map": cluster_risk_map,
        "ae_threshold": ae_threshold,
        "feature_cols": feature_cols,
        "autoencoder": autoencoder
    })

    st.success("Models loaded successfully")

uploaded_data = st.file_uploader("Upload student CSV", type="csv")
if uploaded_data:
    df = pd.read_csv(uploaded_data)
    st.session_state["df"] = df
    st.success("Student data loaded")

if st.button("Run Student Risk Detection"):
    scaler = st.session_state.get("scaler")
    iso = st.session_state.get("iso")
    kmeans = st.session_state.get("kmeans")
    cluster_risk_map = st.session_state.get("cluster_risk_map")
    ae_threshold = st.session_state.get("ae_threshold")
    feature_cols = st.session_state.get("feature_cols")
    autoencoder = st.session_state.get("autoencoder")
    df = st.session_state.get("df")

    if any(v is None for v in [scaler, iso, kmeans, cluster_risk_map, ae_threshold, feature_cols, autoencoder, df]):
        st.error("Upload models and data first")
    else:
        missing = [c for c in feature_cols if c not in df.columns]
        if missing:
            st.error(f"Missing required feature columns in CSV: {missing}")
        else:
            X = df[feature_cols].values
            X_scaled = scaler.transform(X)

            df["iso_score"] = -iso.decision_function(X_scaled)
            df["iso_flag"] = (iso.predict(X_scaled) == -1).astype(int)

            df["cluster_id"] = kmeans.predict(X_scaled)
            df["cluster_risk"] = df["cluster_id"].map(cluster_risk_map).fillna(0)

            X_recon = autoencoder.predict(X_scaled, verbose=0)
            df["ae_error"] = np.mean((X_scaled - X_recon) ** 2, axis=1)
            df["ae_flag"] = (df["ae_error"] > ae_threshold).astype(int)

            df["final_risk_score"] = (
                0.4 * df["cluster_risk"] +
                0.3 * df["iso_flag"] +
                0.3 * df["ae_flag"]
            )
            df["final_at_risk"] = (df["final_risk_score"] >= 0.6).astype(int)

            st.subheader("🚨 At-Risk Students")
            st.dataframe(df[df["final_at_risk"] == 1])

            st.metric("Total Students", len(df))
            st.metric("At-Risk Students", int(df["final_at_risk"].sum()))




#styling
bg_file = "web_bg.png"  # make sure you uploaded this file
bg_base64 = get_base64_of_bin_file(bg_file)
st.markdown(
    f"""
    <style>
    .stApp {{
        background: url("data:image/png;base64,{bg_base64}");
        background-size: cover;
        background-repeat: no-repeat;
        background-attachment: fixed;
        font-family: 'Segoe UI', sans-serif;
        color: #692b39;
    }}
    h1, p, .stMarkdown, .stSubheader, .stMetric label {{
        color: #692b39 !important;
    }}
    .stFileUploader {{
        border: 2px solid #ffa51f;
        border-radius: 8px;
        padding: 0.6em;
        background-color: #692b39;
    }}
    .stFileUploader label p {{
        color: #f4ebd0 !important;
        font-weight: bold;
    }}
    .stFileUploader section {{
        background-color: #692b39 !important;
    }}
    /* Fixes internal "Drag and drop" text and icons */
    [data-testid="stFileUploaderDropzoneInstructions"] div,
    [data-testid="stFileUploaderDropzoneInstructions"] small,
    [data-testid="stFileUploaderDropzoneInstructions"] span,
    [data-testid="stFileUploaderDropzone"] svg {{
        color: #f4ebd0 !important;
        fill: #f4ebd0 !important;
    }}
    /* FIX FOR UPLOADED FILE INFO (Name, Icons, and SIZE) */
    [data-testid="stFileUploaderFileName"],
    [data-testid="stFileUploaderFileData"] div,
    [data-testid="stFileUploaderFileData"] small,
    [data-testid="stFileUploaderFileData"] span,
    [data-testid="stFileUploaderFileData"] svg {{
        color: #f4ebd0 !important;
        fill: #f4ebd0 !important;
    }}
    /* Main Browse Button stays white with black text */
    .stFileUploader button {{
        background-color: #ffffff !important;
        color: #000000 !important;
        transition: 0.3s;
    }}
    .stFileUploader button:hover {{
        background-color: #ffa51f !important;
        color: #000000 !important;
        border: none !important;
    }}
    .stButton>button {{
        background-color: #692b39 !important;
        color: #f4ebd0 !important;   /* <-- beige text */
        border-radius: 8px;
        padding: 0.6em 1.2em;
        font-weight: bold;
        transition: 0.3s;
    }}
    .stButton>button p {{
        color: #f4ebd0 !important;
    }}
    .stButton>button span {{
        color: #f4ebd0 !important;   /* <-- force text color inside span */
    }}
    .stButton>button:hover {{
        background-color: #4a1d28 !important;
        transform: scale(1.05);
    }}
    .stDataFrame {{
        border: 2px solid #692b39;
        border-radius: 8px;
        padding: 0.5em;
        background-color: #FAFAFA;
        color: #692b39;
    }}
    .stMetric {{
        background-color: #692b39;
        border-radius: 8px;
        padding: 0.8em;
        margin: 0.5em 0;
    }}
    .stMetric label p {{
        color: #f4ebd0 !important;
        font-weight: bold;
    }}
    div[data-testid="stMetricValue"] p,
    div[data-testid="stMetricLabel"] p {{
        color: #f4ebd0 !important;
    }}
    </style>
    """,
    unsafe_allow_html=True
)

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
!streamlit run student_risk_detection_app.py